# Run the pilot on a free GPU (Colab / Kaggle)

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

This notebook clones the repo, downloads CIFAR-10-C, and runs the **phase-1
signal check** (teacher + `w0.5` student, fp32, seeds {0,1}, 30 epochs) end to
end: train -> evaluate (ID + CIFAR-10-C) -> geometry -> aggregate. ~15-25 min on
a T4.

The full 18-run pilot is the last cell (longer; see the note there).

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Clone the repo (dev branch = latest) and enter it
%cd /content
![ -d student-teacher-landscape ] || git clone -b dev https://github.com/Iyeba-Kallon/student-teacher-landscape.git
%cd /content/student-teacher-landscape
!git pull --quiet

In [ ]:
# Only extra dependency Colab does not already have. (Keep Colab's own torch.)
!pip install -q pyhessian
import pyhessian, yaml, numpy, pandas, torchvision  # sanity
print('deps OK')

In [ ]:
# CIFAR-10-C (~2.9 GB, ~2-3 min on Colab). Idempotent. CIFAR-10 auto-downloads later.
!python data/download_cifar10c.py --dest data/

In [ ]:
# Phase-1 signal check, end to end (~15-25 min on a T4).
# Geometry uses the locked defaults (2000-example subset, 100 Hutchinson iters).
!bash scripts/phase1.sh

In [ ]:
import pandas as pd
df = pd.read_csv('results/pilot_summary.csv')
df[['run_name','mode','width_mult','seed','id_acc','ood_acc_mean','mce_vs_baseline',
    'adaptive_sharpness','hessian_trace','hessian_top_eigenvalue']]

In [ ]:
# Quick read: does each student beat its teacher on OOD, and is it flatter?
t = df[df['mode']=='teacher'].set_index('seed')
for _, s in df[df['mode']=='student'].iterrows():
    ts = t.loc[s['seed']]
    print(f"seed {s['seed']}  {s['run_name']:24s} "
          f"OOD {s['ood_acc_mean']:.3f} vs {ts['ood_acc_mean']:.3f} "
          f"({'BEATS' if s['ood_acc_mean']>ts['ood_acc_mean'] else 'below':>5}) | "
          f"sharpness {s['adaptive_sharpness']:.3f} vs {ts['adaptive_sharpness']:.3f} "
          f"({'FLATTER' if s['adaptive_sharpness']<ts['adaptive_sharpness'] else 'sharper'})")

In [ ]:
# Save results back to your machine (zip -> download).
!zip -qr results_phase1.zip results/ -x 'results/**/checkpoints/*'
from google.colab import files
files.download('results_phase1.zip')

---
## Full 18-run pilot

If phase-1 shows the pattern (students beat teacher on OOD **and** are flatter),
run the real thing: restore `epochs: 200` in `configs/teacher_fp32.yaml` and
`configs/student_w0.5_fp32.yaml`, then:

```python
!SKIP_DATA=1 bash scripts/run_pilot.sh
```

This is `{teacher, w0.5, w0.25} x {fp32, amp} x seeds {0,1,2}` = 18 runs +
evaluate + geometry. On a single T4 that is **~12-18 h**, which exceeds a Colab
free session. Options:

- **Kaggle** (30 h/week GPU, 12 h sessions) — run it in 2 chunks; `run_pilot.sh`
  / `phase1.sh` skip finished training runs on restart.
- **Persist `results/` to Google Drive** so a killed session resumes:
  ```python
  from google.colab import drive; drive.mount('/content/drive')
  !mkdir -p /content/drive/MyDrive/stl_results && ln -sfn /content/drive/MyDrive/stl_results results
  ```
  then re-run the pilot cell after each reconnect.
- Run **one precision at a time**: `SKIP_AMP=1 ...` first (fp32 half), then the
  AMP configs.

Bring back `results/pilot_summary.csv` + the per-run `summary.json` / `eval.json`
/ `geometry.json` and open `notebooks/analysis.ipynb` on them locally.